In [1]:
%load_ext autoreload
%autoreload 2

%autosave 30

Autosaving every 30 seconds


# CoralNet Source Audit

This notebook audits all CoralNet source folders in S3, checking file structure,
counting images and annotations, and writing results to a parquet file.

**Output**: `s3://dev-datamermaid-sm-sources/dev/coralnet_source_audit_YYYYMMDD.parquet`

## 1. Setup

In [2]:
import os

os.environ["AWS_PROFILE"] = "mermaid-core"

In [3]:
from datetime import UTC, datetime

import boto3
import ibis
import pandas as pd
from botocore.exceptions import ClientError
from great_tables import GT
from tqdm.auto import tqdm

ibis.options.interactive = True

In [4]:
con = ibis.duckdb.connect()
con.raw_sql("""
    INSTALL httpfs;
    LOAD httpfs;
""")
con.raw_sql("CREATE OR REPLACE SECRET s3 (TYPE S3, PROVIDER CREDENTIAL_CHAIN)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
BUCKET = "dev-datamermaid-sm-sources"
PREFIX = "coralnet-public-images"
DATE = "20260423"
OUTPUT_PATH = f"s3://{BUCKET}/dev/coralnet_source_audit_{DATE}.parquet"

s3 = boto3.client("s3")

## 2. Helper Functions

In [6]:
def list_source_folders(s3_client, bucket: str, prefix: str) -> list[int]:
    """List all source folders directly from the target bucket.

    Returns ALL source IDs found with no filtering.
    """
    full_prefix = f"{prefix}/" if not prefix.endswith("/") else prefix
    paginator = s3_client.get_paginator("list_objects_v2")

    source_ids = []
    for page in paginator.paginate(Bucket=bucket, Prefix=full_prefix, Delimiter="/"):
        for common_prefix in page.get("CommonPrefixes", []):
            folder_name = common_prefix["Prefix"].replace(full_prefix, "").rstrip("/")
            if folder_name.startswith("s") and folder_name[1:].isdigit():
                source_ids.append(int(folder_name[1:]))

    return sorted(source_ids)

In [7]:
def check_file_exists(s3_client, bucket: str, key: str) -> bool:
    """Check if a file exists in S3 using HEAD request."""
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "404":
            return False
        raise

In [8]:
def check_prefix_exists(s3_client, bucket: str, prefix: str) -> bool:
    """Check if a prefix (folder) has any objects."""
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=1)
    return response.get("KeyCount", 0) > 0

In [9]:
def count_images_in_s3(s3_client, bucket: str, images_prefix: str) -> int:
    """Count all objects in an images/ folder using pagination."""
    paginator = s3_client.get_paginator("list_objects_v2")
    count = 0
    for page in paginator.paginate(Bucket=bucket, Prefix=images_prefix):
        count += page.get("KeyCount", 0)
    return count

In [10]:
def read_csv_with_ibis(con, s3_uri: str) -> ibis.Table | None:
    """Read a CSV from S3 using ibis. Returns None if file doesn't exist or can't be read."""
    try:
        return con.read_csv(s3_uri)
    except Exception:
        return None


def count_csv_rows(con, s3_uri: str) -> int | None:
    """Count rows in a CSV using ibis. Returns None if file can't be read."""
    try:
        table = con.read_csv(s3_uri)
        return table.count().execute()
    except Exception:
        return None

In [11]:
def audit_source(s3_client, con, bucket: str, source_id: int, prefix: str, audit_ts: datetime) -> dict:
    """Audit a single CoralNet source folder.

    Returns a dictionary with all audit fields. Handles missing files gracefully.
    Uses ibis for CSV reads.
    """
    source_prefix = f"{prefix}/s{source_id}/"
    errors = []

    record = {
        "source_id": source_id,
        "audit_timestamp": audit_ts,
        "has_images_folder": False,
        "has_annotations_csv": False,
        "has_image_list_csv": False,
        "has_labelset_csv": False,
        "has_metadata_csv": False,
        "n_images_s3": 0,
        "n_images_csv": 0,
        "n_annotations": 0,
        "n_unique_images_annotated": 0,
        "n_labels": None,
        "n_metadata_rows": None,
        "annotations_empty": False,
        "image_list_empty": False,
        "labelset_empty": False,
        "metadata_empty": False,
        "is_complete": False,
        "image_count_match": False,
        "errors": [],
    }

    try:
        images_prefix = f"{source_prefix}images/"
        record["has_images_folder"] = check_prefix_exists(s3_client, bucket, images_prefix)
        if record["has_images_folder"]:
            record["n_images_s3"] = count_images_in_s3(s3_client, bucket, images_prefix)
    except Exception as e:
        errors.append(f"Error checking images folder: {e}")

    try:
        annotations_key = f"{source_prefix}annotations.csv"
        record["has_annotations_csv"] = check_file_exists(s3_client, bucket, annotations_key)
        if record["has_annotations_csv"]:
            annotations_uri = f"s3://{bucket}/{annotations_key}"
            count = count_csv_rows(con, annotations_uri)
            if count is not None:
                record["n_annotations"] = count
                record["annotations_empty"] = count == 0
                if count > 0:
                    table = read_csv_with_ibis(con, annotations_uri)
                    if table is not None:
                        cols = table.columns
                        if "Name" in cols:
                            record["n_unique_images_annotated"] = table.Name.nunique().execute()
                        elif "image" in cols:
                            record["n_unique_images_annotated"] = table.image.nunique().execute()
    except Exception as e:
        errors.append(f"Error reading annotations.csv: {e}")

    try:
        image_list_key = f"{source_prefix}image_list.csv"
        record["has_image_list_csv"] = check_file_exists(s3_client, bucket, image_list_key)
        if record["has_image_list_csv"]:
            image_list_uri = f"s3://{bucket}/{image_list_key}"
            count = count_csv_rows(con, image_list_uri)
            if count is not None:
                record["n_images_csv"] = count
                record["image_list_empty"] = count == 0
    except Exception as e:
        errors.append(f"Error reading image_list.csv: {e}")

    try:
        labelset_key = f"{source_prefix}labelset.csv"
        record["has_labelset_csv"] = check_file_exists(s3_client, bucket, labelset_key)
        if record["has_labelset_csv"]:
            labelset_uri = f"s3://{bucket}/{labelset_key}"
            count = count_csv_rows(con, labelset_uri)
            if count is not None:
                record["n_labels"] = count
                record["labelset_empty"] = count == 0
    except Exception as e:
        errors.append(f"Error reading labelset.csv: {e}")

    try:
        metadata_key = f"{source_prefix}metadata.csv"
        record["has_metadata_csv"] = check_file_exists(s3_client, bucket, metadata_key)
        if record["has_metadata_csv"]:
            metadata_uri = f"s3://{bucket}/{metadata_key}"
            count = count_csv_rows(con, metadata_uri)
            if count is not None:
                record["n_metadata_rows"] = count
                record["metadata_empty"] = count == 0
    except Exception as e:
        errors.append(f"Error reading metadata.csv: {e}")

    record["is_complete"] = record["has_images_folder"] and record["has_annotations_csv"] and record["has_image_list_csv"]
    record["image_count_match"] = record["n_images_s3"] == record["n_images_csv"]
    record["errors"] = errors

    return record

## 3. Execute Audit

In [ ]:
source_ids = list_source_folders(s3, BUCKET, PREFIX)
print(f"Found {len(source_ids)} source folders to audit")

In [ ]:
audit_timestamp = datetime.now(UTC)

results = []
for source_id in tqdm(source_ids, desc="Auditing sources"):
    record = audit_source(s3, con, BUCKET, source_id, PREFIX, audit_timestamp)
    results.append(record)

## 4. Convert and Write

In [ ]:
audit_df = pd.DataFrame(results)
audit_df["errors"] = audit_df["errors"].apply(lambda x: x if x else None)

In [ ]:
audit_df.dtypes

In [ ]:
audit_table = ibis.memtable(audit_df)
con.to_parquet(audit_table, OUTPUT_PATH)
print(f"Written audit results to {OUTPUT_PATH}")

## 5. Analysis

In [14]:
audit = con.read_parquet(OUTPUT_PATH)

In [15]:
audit.head()

┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ source_id ┃ audit_timestamp                  ┃ has_images_folder ┃ has_annotations_csv ┃ has_image_list_csv ┃ has_labelset_csv ┃ has_metadata_csv ┃ n_images_s3 ┃ n_images_csv ┃ n_annotations ┃ n_unique_images_annotated ┃ n_labels ┃ is_complete ┃ image_count_match ┃ errors ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ int64     │ timestamp('UTC', 6)              │ boolean           │ boolean             │ boolean            │ boolean          │ boolean          │ int64       │ int64        │ int64         │ int64                     │ float64  │ boolean     │ boolean           │ int32  │
├───────────┼──────────────────────────────────┼───────────────────┼─────────────────────┼────────────────────┼──────────────────┼──────────────────┼─────────────┼──────────────┼───────────────┼───────────────────────────┼──────────┼─────────────┼───────────────────┼────────┤
│        23 │ 2026-04-23 02:07:01.702711+00:00 │ True              │ True                │ True               │ True             │ True             │         750 │          750 │         15000 │                       750 │     20.0 │ True        │ True              │   NULL │
│        57 │ 2026-04-23 02:07:01.702711+00:00 │ True              │ True                │ True               │ True             │ True             │        1681 │         1681 │        168100 │                      1681 │     35.0 │ True        │ True              │   NULL │
│        69 │ 2026-04-23 02:07:01.702711+00:00 │ True              │ True                │ True               │ True             │ False            │         100 │          100 │          4000 │                       100 │      5.0 │ True        │ True              │   NULL │
│        70 │ 2026-04-23 02:07:01.702711+00:00 │ True              │ True                │ True               │ True             │ True             │         300 │          300 │         12000 │                       300 │     16.0 │ True        │ True              │   NULL │
│        82 │ 2026-04-23 02:07:01.702711+00:00 │ True              │ True                │ True               │ True             │ False            │           1 │            1 │             0 │                         0 │     19.0 │ True        │ True              │   NULL │
└───────────┴──────────────────────────────────┴───────────────────┴─────────────────────┴────────────────────┴──────────────────┴──────────────────┴─────────────┴──────────────┴───────────────┴───────────────────────────┴──────────┴─────────────┴───────────────────┴────────┘

In [16]:
summary_data = {
    "Metric": [
        "Total sources",
        "Complete sources",
        "Sources with images folder",
        "Sources with annotations.csv",
        "Sources with image_list.csv",
        "Sources with labelset.csv",
        "Sources with metadata.csv",
        "Sources with image count match",
        "Sources with empty annotations.csv",
        "Sources with empty image_list.csv",
        "Sources with empty labelset.csv",
        "Sources with empty metadata.csv",
    ],
    "Count": [
        audit.count().execute(),
        audit.filter(audit.is_complete).count().execute(),
        audit.filter(audit.has_images_folder).count().execute(),
        audit.filter(audit.has_annotations_csv).count().execute(),
        audit.filter(audit.has_image_list_csv).count().execute(),
        audit.filter(audit.has_labelset_csv).count().execute(),
        audit.filter(audit.has_metadata_csv).count().execute(),
        audit.filter(audit.image_count_match).count().execute(),
        audit.filter(audit.annotations_empty).count().execute(),
        audit.filter(audit.image_list_empty).count().execute(),
        audit.filter(audit.labelset_empty).count().execute(),
        audit.filter(audit.metadata_empty).count().execute(),
    ],
}
summary_df = pd.DataFrame(summary_data)
total = summary_df.loc[0, "Count"]
summary_df["Percentage"] = (summary_df["Count"] / total * 100).round(1)

In [17]:
GT(summary_df).tab_header(title="CoralNet Source Audit Summary")

GT(_tbl_data=                           Metric  Count  Percentage
0                   Total sources   1509       100.0
1                Complete sources   1461        96.8
2      Sources with images folder   1461        96.8
3    Sources with annotations.csv   1463        97.0
4     Sources with image_list.csv   1463        97.0
5       Sources with labelset.csv   1468        97.3
6       Sources with metadata.csv    979        64.9
7  Sources with image count match   1468        97.3, _body=<great_tables._gt_data.Body object at 0x1377ee060>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='right', column_width=None), ColInfo(var='Percentage', type=<ColInfoTypeEnum.default: 1>, column_label='Percentage', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x136f6c650>, _spanners=Spanners([]), _heading=Heading(title='CoralNet Source Audit Summary', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x137946b10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x137947f20>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x137947ec0>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(sc

In [18]:
totals = audit.aggregate(
    total_images_s3=audit.n_images_s3.sum(),
    total_images_csv=audit.n_images_csv.sum(),
    total_annotations=audit.n_annotations.sum(),
    total_unique_annotated=audit.n_unique_images_annotated.sum(),
    total_labels=audit.n_labels.sum(),
    total_metadata_rows=audit.n_metadata_rows.sum(),
).execute()

totals_df = totals.T.reset_index()
totals_df.columns = ["Metric", "Value"]

In [19]:
GT(totals_df).tab_header(title="Total Resource Counts")

GT(_tbl_data=                   Metric     Value
0         total_images_s3   1101071
1        total_images_csv    670794
2       total_annotations  35200570
3  total_unique_annotated   1145143, _body=<great_tables._gt_data.Body object at 0x1379924e0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1374ad940>, _spanners=Spanners([]), _heading=Heading(title='Total Resource Counts', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x137992180>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x137992090>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x137991fa0>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_color=OptionsInfo(scss=True, category='heading', type='value', value=None), heading_align=OptionsInfo(scss=True, category='heading', type='value', value='center'), heading_title_font_size=OptionsInfo(scss=True, category='heading', type='px', value='125%'), heading_ti

In [20]:
incomplete = (
    audit.filter(~audit.is_complete)
    .select(
        "source_id",
        "has_images_folder",
        "has_annotations_csv",
        "has_image_list_csv",
    )
    .order_by("source_id")
)

In [21]:
incomplete_df = incomplete.execute()
if len(incomplete_df) > 0:
    GT(incomplete_df).tab_header(title="Incomplete Sources")
else:
    print("All sources are complete!")

In [22]:
mismatch = (
    audit.filter(audit.is_complete & ~audit.image_count_match)
    .select(
        "source_id",
        "n_images_s3",
        "n_images_csv",
    )
    .order_by("source_id")
)

In [23]:
mismatch_df = mismatch.execute()
if len(mismatch_df) > 0:
    mismatch_df["diff"] = mismatch_df["n_images_s3"] - mismatch_df["n_images_csv"]
    GT(mismatch_df).tab_header(title="Image Count Mismatches")
else:
    print("All image counts match!")

## 6. Visualizations

In [25]:
completeness_counts = (
    audit_df[["has_images_folder", "has_annotations_csv", "has_image_list_csv", "has_labelset_csv", "has_metadata_csv"]].sum().reset_index()
)
completeness_counts.columns = ["file_type", "count"]
completeness_counts["file_type"] = completeness_counts["file_type"].str.replace("has_", "").str.replace("_", " ")

NameError: name 'audit_df' is not defined

In [ ]:
completeness_counts.hvplot.bar(
    x="file_type",
    y="count",
    title="Sources by File Availability",
    xlabel="File Type",
    ylabel="Number of Sources",
    rot=45,
    height=400,
    width=600,
)

In [ ]:
audit_df[audit_df["n_images_s3"] > 0].hvplot.hist(
    "n_images_s3",
    bins=50,
    title="Distribution of Image Counts per Source",
    xlabel="Number of Images",
    ylabel="Number of Sources",
    height=400,
    width=600,
)

In [ ]:
complete_df = audit_df[audit_df["is_complete"]]
complete_df.hvplot.scatter(
    x="n_images_s3",
    y="n_annotations",
    title="Images vs Annotations per Source",
    xlabel="Number of Images",
    ylabel="Number of Annotations",
    hover_cols=["source_id"],
    height=400,
    width=600,
)